In [11]:
%%writefile src/business_entity_resolution/features.py

import pandas as pd
from rapidfuzz.fuzz import ratio


def similarity(value1, value2):

    if pd.isna(value1) or pd.isna(value2):
        return 0.0

    value1 = str(value1)
    value2 = str(value2)

    if not value1 or not value2:
        return 0.0

    return ratio(value1, value2) / 100.0


def calculate_pair_features(record1, record2):

    name_score = similarity(
        record1.get("normalized_name", ""),
        record2.get("normalized_name", "")
    )

    address_score = similarity(
        record1.get("normalized_address", ""),
        record2.get("normalized_address", "")
    )

    country_score = similarity(
        record1.get("normalized_country", ""),
        record2.get("normalized_country", "")
    )

    return {
        "name_similarity": name_score,
        "address_similarity": address_score,
        "country_similarity": country_score,
    }


%%writefile src/business_entity_resolution/model.py

import pandas as pd

from sklearn.linear_model import LogisticRegression


FEATURE_COLUMNS = [
    "name_similarity",
    "address_similarity",
    "country_similarity",
]


def create_weighted_score(features):

    return (
        0.60 * features["name_similarity"]
        + 0.30 * features["address_similarity"]
        + 0.10 * features["country_similarity"]
    )


def train_model(features, labels):

    X = features[FEATURE_COLUMNS]
    y = labels

    model = LogisticRegression(
        max_iter=1000
    )

    model.fit(X, y)

    return model


def predict_scores(model, features):

    X = features[FEATURE_COLUMNS]

    return model.predict_proba(X)[:, 1]

Overwriting src/business_entity_resolution/features.py
